## Twin NPM Peaks - Verified and Nuanced

Both peak years' aggregate NPMs are cross-verified against Project 1 SQL:
- 2011 aggregate NPM: 20.23% (matched Project 1)
- 2021 aggregate NPM: 21.17% (matched Project 1)

Same headline number, but the sector composition tells a more nuanced 
story than "broad recovery vs concentrated tech":

- Technology was already extraordinarily profitable in 2011 (27.2%). It 
  stayed elevated in 2021 (29.3%)  a small contributor to the increase.
- Banking margins doubled (13.0% → 29.1%), likely reflecting AIG's post-
  bailout normalisation completing.
- Food & Beverage expanded by 12 percentage points (MCD's operational 
  leverage + recovery pricing).
- Manufacturing collapsed into losses (PCG wildfire liabilities in 
  2018-2021).
- Both years happen to have exactly 11 reporting companies: PYPL absent 
  in 2011 (pre-IPO 2014), SHLDQ absent in 2021 (bankrupt 2018).


In [25]:
# Sanity check on Revenue magnitude
print("Revenue sample stats:")
print(df["Revenue"].describe())
print()

# AAPL 2022 should be around $394,000M
print(df[(df["Company"] == "AAPL") & (df["Year"] == 2022)][["Revenue", "Net Income"]])

Revenue sample stats:
count       159.000000
mean      75314.400767
std       90631.174372
min        3326.445000
25%       22782.550000
50%       45992.040000
75%       76648.000000
max      513983.000000
Name: Revenue, dtype: float64

    Revenue  Net Income
0  394328.0     99803.0


In [24]:
# Overall aggregate NPM for 2011 and 2021 (not per-sector)
peak_check = df[df["Year"].isin([2011, 2021])].groupby("Year").agg(
    total_revenue=("Revenue", "sum"),
    total_net_income=("Net Income", "sum"),
    company_count=("Company", "nunique")
).reset_index()

peak_check["aggregate_npm_pct"] = (
    peak_check["total_net_income"] / peak_check["total_revenue"] * 100
).round(2)

peak_check

,Year,total_revenue,total_net_income,company_count,aggregate_npm_pct
0,2011,520314.129,105264.226,11,20.23
1,2021,1508525.590,319285.463,11,21.17


In [ ]:
# The Twin Net Profit Margin(NPM) Peaks Finding
# Which sectors contributed to each peak?
peak_years = df[df["Year"].isin([2011, 2021])]

sector_npm_by_peak = peak_years.groupby(["sector", "Year"]).agg(
    total_revenue=("Revenue", "sum"),
    total_net_income=("Net Income", "sum"),
).reset_index()

sector_npm_by_peak["npm_pct"] = (
    sector_npm_by_peak["total_net_income"] / sector_npm_by_peak["total_revenue"] * 100
).round(1)

# This introduces pivot a new pandas concept. 
# Takes long-format data (rows per sector-year) and reshapes to wide format (rows per sector, columns per year). 
# Section 13 of your PDF references this.

# Pivot for readability
npm_by_peak = sector_npm_by_peak.pivot(
    index="sector", 
    columns="Year", 
    values="npm_pct"
)
npm_by_peak.columns = ["npm_2011", "npm_2021"]
npm_by_peak["change_pp"] = (npm_by_peak["npm_2021"] - npm_by_peak["npm_2011"]).round(1)
npm_by_peak = npm_by_peak.sort_values("npm_2021", ascending=False)
npm_by_peak

,npm_2011,npm_2021,change_pp
sector,,,
Food & Beverage,20.4,32.5,12.1
Technology,27.2,29.3,2.1
Banking,13.0,29.1,16.1
Electronics,22.9,25.3,2.4
FinTech,NaN,16.4,NaN
Logistics,1.3,7.1,5.8
Manufacturing,5.6,-0.5,-6.1
Finance,0.3,NaN,NaN


## Loss-Year Clustering

- 2011 was the dataset's "everyone profitable" year zero loss-makers
- Cluster years (2012, 2014, 2017, 2018) reflect individual company troughs, 
  not systemic crisis
- COVID 2020 had only 2 loss-makers, reinforcing secular-growth dominance
- No year exceeded 3 companies in losses out of 11-12 reporting

In [22]:
# Loss Year Clustering
# Companies with losses per year
# SELECT year, 
#        COUNT(*) AS companies_reporting,
#        SUM(CASE WHEN net_income < 0 THEN 1 ELSE 0 END) AS companies_with_losses,
#        SUM(net_income) AS aggregate_net_income
# FROM financials_raw
# WHERE year BETWEEN 2009 AND 2022
# GROUP BY year
# ORDER BY year;
# Create a boolean/int column: 1 if loss, 0 if profit
df["is_loss"] = (df["Net Income"] < 0).astype(int)

loss_clustering = df.groupby("Year").agg(
    companies_reporting=("Company", "nunique"),
    companies_with_losses=("is_loss", "sum"),
    aggregate_net_income=("Net Income", "sum"),
).reset_index()

loss_clustering["loss_ratio"] = (
    loss_clustering["companies_with_losses"] / loss_clustering["companies_reporting"] * 100
).round(1)

loss_clustering

,Year,companies_reporting,companies_with_losses,aggregate_net_income,loss_ratio
0,2009,11,2,33642.8290,18.2
1,2010,11,1,69186.4230,9.1
2,2011,11,0,105264.2260,0.0
3,2012,11,3,85923.8010,27.3
4,2013,11,1,97490.9928,9.1
5,2014,12,3,100202.0284,25.0
6,2015,12,2,100709.9040,16.7
7,2016,12,2,106880.6300,16.7
8,2017,12,3,98653.2270,25.0
9,2018,12,3,143613.0110,25.0


In [20]:
# Compute Sector CAGR(Compound Annual Growth Rate)

# Step 1: find each sector's year bounds
sector_bounds = sector_year_revenue.groupby("sector").agg(
    first_year=("Year", "min"),
    last_year=("Year", "max"),
).reset_index()

# Step 2: get first-year and last-year revenue for each sector via merge
sector_first_rev = sector_year_revenue.merge(
    sector_bounds[["sector", "first_year"]],
    left_on=["sector", "Year"],
    right_on=["sector", "first_year"],
)[["sector", "sector_revenue"]].rename(columns={"sector_revenue": "first_revenue"})

sector_last_rev = sector_year_revenue.merge(
    sector_bounds[["sector", "last_year"]],
    left_on=["sector", "Year"],
    right_on=["sector", "last_year"],
)[["sector", "sector_revenue"]].rename(columns={"sector_revenue": "last_revenue"})

# Step 3: combine and compute CAGR
sector_cagr = (
    sector_bounds
    .merge(sector_first_rev, on="sector")
    .merge(sector_last_rev, on="sector")
)
sector_cagr["years"] = sector_cagr["last_year"] - sector_cagr["first_year"]
sector_cagr["cagr_pct"] = (
    (sector_cagr["last_revenue"] / sector_cagr["first_revenue"]) ** (1 / sector_cagr["years"]) - 1
) * 100

sector_cagr = sector_cagr.sort_values("cagr_pct", ascending=False).round(2)
sector_cagr[["sector", "first_year", "last_year", "cagr_pct"]]

,sector,first_year,last_year,cagr_pct
5,Logistics,2009,2022,26.38
2,FinTech,2014,2022,16.65
7,Technology,2009,2022,16.15
1,Electronics,2009,2022,6.74
6,Manufacturing,2009,2022,3.77
4,Food & Beverage,2009,2022,0.15
0,Banking,2009,2022,-3.02
3,Finance,2009,2018,-10.81


In [18]:
# Sector Totals at 2009 and 2022.
# For each sector, get total revenue at 2009 (or first available year) and 2022
sector_year_revenue = df.groupby(["sector", "Year"]).agg(
    sector_revenue=("Revenue", "sum"),
    company_count=("Company", "nunique")
).reset_index()

sector_year_revenue.head(10)

,sector,Year,sector_revenue,company_count
0,Banking,2009,45992.04,1
1,Banking,2010,49030.07,1
2,Banking,2011,48866.82,1
3,Banking,2012,46036.06,1
4,Banking,2013,43712.69,1
5,Banking,2014,41677.15,1
6,Banking,2015,38919.16,1
7,Banking,2016,29072.54,1
8,Banking,2017,27162.75,1
9,Banking,2018,28212.33,1


## Aggregate Revenue Trend (2009-2022)

- 2009: $392B → 2022: $1,639B (4.2× multiple)
- Never declined YoY, even in COVID 2020 (+15.3%)
- Lowest growth year: 2016 (+1.1%); strongest: 2021 (+25.1%)
- Twin NPM peaks: 2011 (20.2%) and 2021 (21.2%) — different drivers

In [15]:
# Add Aggregate Net Profit Margin (NPM) (Ratio not Sum)

yearly_revenue["aggregate_npm_pct"] = (
    yearly_revenue["total_net_income"] / yearly_revenue["total_revenue"] * 100
).round(1)

yearly_revenue[["Year", "total_revenue", "total_net_income", "aggregate_npm_pct", "revenue_yoy_pct"]]

,Year,total_revenue,total_net_income,aggregate_npm_pct,revenue_yoy_pct
0,2009,392406.599,33642.8290,8.6,NaN
1,2010,441318.115,69186.4230,15.7,12.5
2,2011,520314.129,105264.2260,20.2,17.9
3,2012,596125.990,85923.8010,14.4,14.6
4,2013,631862.549,97490.9928,15.4,6.0
5,2014,679444.450,100202.0284,14.7,7.5
6,2015,749265.160,100709.9040,13.4,10.3
7,2016,757164.440,106880.6300,14.1,1.1
8,2017,836067.150,98653.2270,11.8,10.4
9,2018,971994.230,143613.0110,14.8,16.3


In [14]:
# Add Year on Year Growth Column
yearly_revenue["revenue_yoy_pct"] = (
    (yearly_revenue["total_revenue"] / yearly_revenue["total_revenue"].shift(1) - 1) * 100
).round(1)

yearly_revenue["net_income_yoy_pct"] = (
    (yearly_revenue["total_net_income"] / yearly_revenue["total_net_income"].shift(1) - 1) * 100
).round(1)

yearly_revenue["market_cap_yoy_pct"] = (
    (yearly_revenue["total_market_cap"] / yearly_revenue["total_market_cap"].shift(1) - 1) * 100
).round(1)

yearly_revenue

,Year,total_revenue,total_net_income,total_market_cap,company_count,revenue_yoy_pct,net_income_yoy_pct,market_cap_yoy_pct
0,2009,392406.599,33642.8290,986.69,11,NaN,NaN,NaN
1,2010,441318.115,69186.4230,1113.79,11,12.5,105.6,12.9
2,2011,520314.129,105264.2260,1199.71,11,17.9,52.1,7.7
3,2012,596125.990,85923.8010,1403.90,11,14.6,-18.4,17.0
4,2013,631862.549,97490.9928,1761.32,11,6.0,13.5,25.5
5,2014,679444.450,100202.0284,1992.01,12,7.5,2.8,13.1
6,2015,749265.160,100709.9040,2357.17,12,10.3,0.5,18.3
7,2016,757164.440,106880.6300,2539.61,12,1.1,6.1,7.7
8,2017,836067.150,98653.2270,3489.49,12,10.4,-7.7,37.4
9,2018,971994.230,143613.0110,3598.13,12,16.3,45.6,3.1


In [ ]:
# Aggregate Revenue by Year, SQL Version

# SELECT year, SUM(revenue) AS total_revenue
# FROM financials_raw  
# WHERE year BETWEEN 2009 AND 2022
# GROUP BY year
# ORDER BY year;

yearly_revenue = df.groupby("Year").agg(
    total_revenue=("Revenue", "sum"),
    total_net_income=("Net Income", "sum"),
    total_market_cap=("Market Cap(in B USD)", "sum"),
    company_count=("Company", "nunique")
).reset_index() 
# with reset_index the year becomes a seperate column and not the index itself.

yearly_revenue

,Year,total_revenue,total_net_income,total_market_cap,company_count
0,2009,392406.599,33642.8290,986.69,11
1,2010,441318.115,69186.4230,1113.79,11
2,2011,520314.129,105264.2260,1199.71,11
3,2012,596125.990,85923.8010,1403.90,11
4,2013,631862.549,97490.9928,1761.32,11
5,2014,679444.450,100202.0284,1992.01,12
6,2015,749265.160,100709.9040,2357.17,12
7,2016,757164.440,106880.6300,2539.61,12
8,2017,836067.150,98653.2270,3489.49,12
9,2018,971994.230,143613.0110,3598.13,12


In [11]:
# Verify
print(df.shape)
print(df["sector"].isnull().sum())   # Should be 0

(159, 24)
14


In [7]:
# Filter to analysis window
df = df[(df["Year"] >= 2009) & (df["Year"] <= 2022)]

In [6]:
# Apply sector mapping (from yesterday)
sector_map = {
    "IT": "Technology",
    "LOGI": "Logistics",
    "FOOD": "Food & Beverage",
    "BANK": "Banking",
    "ELEC": "Electronics",
    "FinTech": "FinTech",
    "Finance": "Finance",
    "Manufacturing": "Manufacturing",
}
df["sector"] = df["Category"].map(sector_map)

In [5]:
import pandas as pd
from pathlib import Path

SCRIPT_DIR = Path().resolve()
df = pd.read_csv(SCRIPT_DIR / "data" / "Financial Statements.csv")
df.columns = df.columns.str.strip()